# Analizador de Similitud de PDFs

Este notebook proporciona un **recorrido interactivo, paso a paso**, del pipeline completo de analisis de similitud de PDFs.  
Importa desde el paquete `src/` y llama a las mismas funciones que usa `main.py`, pero muestra todos los resultados en linea para exploracion y presentacion.

### Resumen del pipeline
1. **Cargar PDFs** — extraer texto crudo de dos documentos PDF.
2. **Extraer palabras clave** — identificar los terminos mas representativos usando KeyBERT.
3. **Calcular similitud** — puntuaciones Jaccard (lexica) + Semantica (significado).
4. **Visualizar** — diagrama de Venn de solapamiento de palabras clave y grafico de barras.
5. **Exportar reporte** — guardar un resumen en texto plano en disco.

In [ ]:
# Configuracion: agregar la raiz del proyecto a sys.path e importar todos los modulos
import sys
import logging
from pathlib import Path

# Asegurar que la raiz del proyecto este en el path para que los imports `src.*` funcionen
PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> raiz del proyecto
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Configurar logging para mostrar mensajes INFO en linea
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(name)s — %(message)s",
    force=True,
)

import dataclasses
import pandas as pd
from IPython.display import Image, display

from src.extractor import extract_text, get_text_preview
from src.keywords import extract_keywords, extract_keywords_with_scores, filter_keywords
from src.similarity import compute_similarity
from src.visualization import plot_venn_diagram, plot_score_bars, generate_report

# NOTA: Actualiza PDF_A y PDF_B en la siguiente celda para que apunten
# a tus archivos PDF reales antes de ejecutar el notebook completo.
print("Configuracion completa — todos los modulos importados.")

## Paso 1 — Cargar PDFs

Apunta `PDF_A` y `PDF_B` a los dos documentos que deseas comparar.  
El extractor lee cada pagina y normaliza el texto (espacios en blanco, saltos de linea).

In [ ]:
# Definir rutas de los PDFs y extraer texto de cada documento
PDF_A = PROJECT_ROOT / "pdfs" / "articulo1.pdf"   # <-- ACTUALIZAR con tu archivo
PDF_B = PROJECT_ROOT / "pdfs" / "articulo2.pdf"   # <-- ACTUALIZAR con tu archivo

text_a = extract_text(str(PDF_A))
text_b = extract_text(str(PDF_B))

print(f"PDF A ({PDF_A.name}): {len(text_a):,} caracteres")
print(get_text_preview(text_a, max_chars=300))
print()
print(f"PDF B ({PDF_B.name}): {len(text_b):,} caracteres")
print(get_text_preview(text_b, max_chars=300))

## Paso 2 — Extraer palabras clave

KeyBERT (respaldado por un sentence-transformer multilingue) selecciona las palabras clave  
y frases mas representativas de cada documento usando Max-Marginal-Relevance para diversidad.

In [ ]:
# Extraer palabras clave con puntuaciones y mostrar como DataFrames lado a lado
TOP_N = 25

kw_scores_a = extract_keywords_with_scores(text_a, top_n=TOP_N)
kw_scores_b = extract_keywords_with_scores(text_b, top_n=TOP_N)

df_a = pd.DataFrame(kw_scores_a, columns=["palabra_clave", "puntuacion"])
df_a["puntuacion"] = df_a["puntuacion"].round(4)

df_b = pd.DataFrame(kw_scores_b, columns=["palabra_clave", "puntuacion"])
df_b["puntuacion"] = df_b["puntuacion"].round(4)

print(f"Palabras clave de {PDF_A.name}:")
display(df_a)

print(f"\nPalabras clave de {PDF_B.name}:")
display(df_b)

# Construir listas simples de palabras clave (sin puntuaciones) para el calculo de similitud
keywords_a = filter_keywords([kw for kw, _ in kw_scores_a])
keywords_b = filter_keywords([kw for kw, _ in kw_scores_b])

## Paso 3 — Calcular similitud

Se calculan y combinan dos metricas complementarias:
- **Jaccard** — solapamiento lexico de conjuntos de palabras clave.
- **Semantica** — similitud coseno de embeddings de sentence-transformer.

In [ ]:
# Calcular similitud y mostrar todos los campos del resultado
result = compute_similarity(
    keywords_a=keywords_a,
    keywords_b=keywords_b,
    text_a=text_a,
    text_b=text_b,
    jaccard_weight=0.4,
    semantic_weight=0.6,
)

print("Resultado de Similitud")
print("=" * 50)
for field in dataclasses.fields(result):
    value = getattr(result, field.name)
    print(f"  {field.name:<20s} : {value}")

## Paso 4 — Visualizaciones

Generar un diagrama de Venn (solapamiento de palabras clave) y un grafico de barras horizontales (comparacion de puntuaciones).  
Ambos se guardan en `outputs/` y se muestran en linea a continuacion.

In [ ]:
# Generar y mostrar diagrama de Venn + grafico de barras de puntuaciones
LABEL_A = PDF_A.stem
LABEL_B = PDF_B.stem
OUTPUT_DIR = PROJECT_ROOT / "outputs"

venn_path = plot_venn_diagram(
    result,
    label_a=LABEL_A,
    label_b=LABEL_B,
    output_path=str(OUTPUT_DIR / "venn.png"),
)
print("Diagrama de Venn:")
display(Image(filename=venn_path))

bars_path = plot_score_bars(
    result,
    label_a=LABEL_A,
    label_b=LABEL_B,
    output_path=str(OUTPUT_DIR / "scores.png"),
)
print("\nGrafico de barras de puntuaciones:")
display(Image(filename=bars_path))

## Paso 5 — Exportar reporte

Guardar un reporte resumen en texto plano en disco y mostrar su contenido.

In [ ]:
# Generar reporte de texto e imprimir su contenido
report_path = generate_report(
    result,
    label_a=LABEL_A,
    label_b=LABEL_B,
    output_path=str(OUTPUT_DIR / "report.txt"),
)

report_text = Path(report_path).read_text(encoding="utf-8")
print(report_text)